# Resolver report

Data comes from `reports/resolver.duckdb`, refreshed by `scripts/report.sh` before this notebook opens.

Run all cells (**Kernel → Restart & Run All**) to see the current state of your benchmark runs.

In [ ]:
from pathlib import Path
import os
import duckdb


def _repo_root():
    here = Path(os.getcwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "go.mod").exists():
            return p
    raise RuntimeError("repo root (go.mod) not found — run via scripts/report.sh")


ROOT = _repo_root()
DB = ROOT / "reports" / "resolver.duckdb"
if not DB.exists():
    raise RuntimeError(f"{DB} missing — run scripts/report.sh first")

con = duckdb.connect(str(DB), read_only=True)
print(f"connected: {DB}")


## What's in the database?

In [ ]:
con.sql("SHOW TABLES").to_df()

## Runs — one row per benchmark run

`virtual_model` is the harness-facing name (what you pass as `--model`); `real_model` is the backing model from the sidecar `run-config.yaml`. `p95_ms` is the 95th-percentile per-query latency. v2.1 no longer surfaces a monolithic `overall` verdict — see the role-coverage heat-map below.

In [ ]:
con.sql("""
    SELECT run_id, model AS virtual_model, cfg_real_model AS real_model,
           correct_count, query_count, p95_ms
    FROM run_summary
    ORDER BY run_id DESC
""").to_df()

## Role-coverage heat-map

Rows: `real-model (virtual)` so you can see each variant without losing the real-model grouping. Columns: one of the 13 v2.1 roles. Cells: most recent verdict (PASS green, FAIL red, ERROR amber, missing transparent). Source: the `role_coverage` view.

In [ ]:
# Role-coverage heat-map: rows = real-model (virtual), columns = role,
# cells = the *most recent* verdict for that pair (PASS green, FAIL red,
# ERROR amber, missing transparent). Source is the v2.1 `role_coverage`
# view produced by the aggregator (internal/aggregate/schema.go).
import pandas as pd

df = con.sql("""
    SELECT COALESCE(
               NULLIF(rc.resolved_real_model, ''),
               r.model
           ) || ' (' || r.model || ')' AS model_label,
           rc.role, rc.verdict, rc.run_id
    FROM role_coverage rc
    JOIN runs r ON r.run_id = rc.run_id
    ORDER BY rc.run_id DESC
""").to_df()

# Most recent verdict per (model_label, role).
recent = df.drop_duplicates(subset=["model_label", "role"], keep="first")
pivot = recent.pivot(index="model_label", columns="role", values="verdict")


def _color(v):
    # transparent background preserves legibility on dark-mode Jupyter.
    if v == "PASS":
        return "background-color: #4caf50; color: white"
    if v == "FAIL":
        return "background-color: #e53935; color: white"
    if v == "ERROR":
        return "background-color: #fb8c00; color: white"
    return "background-color: transparent"


pivot.style.map(_color).set_caption("Role-coverage heat-map (most recent verdict)")


### Per-role thresholds vs observed

Which roles are gated, what threshold each run was scored against, and
whether the gate cleared. `threshold_met` is the authoritative gate
decision — the color in the heat-map above is just a summary.

In [ ]:
con.sql("""
    SELECT run_id, role, verdict, threshold_met, threshold,
           scenario_count_expected AS expected,
           scenario_count_observed AS observed
    FROM role_coverage
    ORDER BY run_id DESC, role
""").to_df()

## Per-query results

Flat view: one row per (run, scenario). Limited to the most recent 50 rows; edit the SQL to widen.

In [ ]:
con.sql("""
    SELECT run_id, model AS virtual_model,
           cfg_real_model AS real_model,
           scenario_id, score, elapsed_ms
    FROM comparison
    ORDER BY run_id DESC, scenario_id
    LIMIT 50
""").to_df()

## Community benchmarks

Reference scores from the broader LLM ecosystem. `model_key` is the normalised form used to join against the real model name.

In [ ]:
con.sql("""
    SELECT model, model_key, benchmark, metric, value, as_of
    FROM community_benchmarks
    ORDER BY model, benchmark
""").to_df()

### Model × benchmark matrix

Pivoted view — rows are normalised `model_key`, columns are `benchmark/metric` pairs.

In [ ]:
# Pivot: one row per model_key (normalised name so quant / casing /
# org-prefix variants collapse), one column per benchmark/metric pair,
# values are the published reference scores.
df = con.sql("""
    SELECT model_key,
           benchmark || '/' || metric AS bench_metric,
           value
    FROM community_benchmarks
""").to_df()

df.pivot_table(index="model_key", columns="bench_metric", values="value", aggfunc="first")


## Next

- Drill into a single run in **`reproducibility.ipynb`**.
- Write your own SQL below — `con` is a read-only DuckDB connection and `.to_df()` renders a pandas DataFrame.
- Re-run `scripts/report.sh` after new benchmark runs to refresh.